<a href="https://colab.research.google.com/github/dr-dlr/drlab/blob/hackathon/ETL_hackathon.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# INTRODUCCIÓN

El presente notebook consta del proceso de extracción, transformación y carga de los datasets de una empresa de la rama fintech, la finalidad es preparar los sets para su posterior análisis exploratorio y entrenamiento de modelos del equipo de ciencia de datos.

# Importación de librerías

Se llama a las librerías necesarias para realizar la visualización de datos y su manipulación

In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib as plt

# Revisión y limpieza de datos de la tabla Transacciones

In [ ]:
# Se asigna una variable al dataset
transacciones = pd.read_csv('/content/transactions.csv')

In [ ]:
# Se revisan brevemente las columnas
transacciones.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 119465 entries, 0 to 119464
Data columns (total 13 columns):
 #   Column                   Non-Null Count   Dtype  
---  ------                   --------------   -----  
 0   transaction_id           119465 non-null  int64  
 1   user_id                  119465 non-null  int64  
 2   fecha_transaccion        119465 non-null  object 
 3   descripcion_transaccion  119465 non-null  object 
 4   monto_transaccion        119465 non-null  float64
 5   medio_pago               119465 non-null  object 
 6   edad                     119465 non-null  int64  
 7   sexo                     119465 non-null  object 
 8   ingreso_mensual          119465 non-null  int64  
 9   linea_credito            119465 non-null  int64  
 10  credito_utilizado        119465 non-null  int64  
 11  frecuencia_ahorro        119465 non-null  object 
 12  categoria_gasto          119465 non-null  object 
dtypes: float64(1), int64(6), object(6)
memory usage: 11.8+ MB


In [ ]:
# Se revisa brevemente el contenido de la tabla de transacciones
transacciones.sample(5)

,transaction_id,user_id,fecha_transaccion,descripcion_transaccion,monto_transaccion,medio_pago,edad,sexo,ingreso_mensual,linea_credito,credito_utilizado,frecuencia_ahorro,categoria_gasto
36628,36629,918,2025-05-26,Netflix,156.67,Credito,22,M,20311,120000,92383,Media,Entretenimiento
96559,96560,2424,2025-07-09,Netflix,2032.42,Efectivo,30,M,60244,50000,12833,Media,Entretenimiento
55315,55316,1388,2025-03-21,Telcel,1554.89,Transferencia,25,F,67433,15000,14513,Media,Servicios
115259,115260,2894,2025-11-11,DiDi,1604.80,Transferencia,51,F,34063,30000,19294,Media,Transporte
43176,43177,1083,2025-05-09,CFE,1942.94,Efectivo,36,M,62814,50000,1282,Baja,Servicios


Se trata de una tabla de 13 columnas y 119,465 filas. Las columnas se dividen entre el identificador de la transacción, identificador de cliente, fecha de transacción, descripción de transacción, monto de transacción, forma de pago, edad del cliente, sexo del cliente, ingreso mensual del cliente expresado en pesos mexicanos (MXN), línea de crédito del cliente en pesos mexicanos, el crédito ocupado por el cliente en pesos mexicanos, la frecuencia de ahorro del cliente presumiblemente determinada de acuerdo con el resultado entre sus ingresos, el crédito utilizado y otras erogaciones del cliente no rastreables de forma digital como retiros de efectivo.

In [ ]:
# Revisando las categorías en 'categoria_gasto'
print (transacciones['categoria_gasto'].unique())
print (f'Se encuentran ausentes dos categorías: "Vivienda y Educación')

['Alimentacion' 'Salud' 'Transporte' 'Compras' 'Servicios'
 'Entretenimiento']
Se encuentran ausentes dos categorías: "Vivienda y Educación


Se da cuenta de la falta de dos categorías planteadas por la empresa, la categoría de educación y la de vivienda, a continuación se añadirán para efectos de clasificación correcta del posterior modelo de clasificación de procesamiento de lenguaje natural

In [ ]:
# Añadiendo las categorías que faltaban de acuerdo con lo planteado por la empresa (educación y vivienda)

import numpy as np
import random

unique_users = transacciones['user_id'].unique()

new_rows = []

for user in unique_users:
    user_info = transacciones[transacciones['user_id'] == user].iloc[0]

    # 1. Transacción de Vivienda
    new_rows.append({
        'transaction_id': 0, # Se resetearán después
        'user_id': user,
        'fecha_transaccion': '2025-06-15',
        'descripcion_transaccion': random.choice(['Home Depot', 'Interceramic']),
        'monto_transaccion': round(random.uniform(500, 5000), 2),
        'medio_pago': random.choice(['Debito', 'Credito', 'Transferencia']),
        'edad': user_info['edad'],
        'sexo': user_info['sexo'],
        'ingreso_mensual': user_info['ingreso_mensual'],
        'linea_credito': user_info['linea_credito'],
        'credito_utilizado': user_info['credito_utilizado'],
        'frecuencia_ahorro': user_info['frecuencia_ahorro'],
        'categoria_gasto': 'Vivienda'
    })

    # 2. Transacción de Educación
    new_rows.append({
        'transaction_id': 0,
        'user_id': user,
        'fecha_transaccion': '2025-07-20',
        'descripcion_transaccion': random.choice(['Librerías Gandhi', 'Proveedora Escolar']),
        'monto_transaccion': round(random.uniform(200, 1500), 2),
        'medio_pago': random.choice(['Debito', 'Credito', 'Efectivo']),
        'edad': user_info['edad'],
        'sexo': user_info['sexo'],
        'ingreso_mensual': user_info['ingreso_mensual'],
        'linea_credito': user_info['linea_credito'],
        'credito_utilizado': user_info['credito_utilizado'],
        'frecuencia_ahorro': user_info['frecuencia_ahorro'],
        'categoria_gasto': 'Educacion'
    })

# Crear DataFrame con nuevas filas y concatenar
df_nuevos = pd.DataFrame(new_rows)
transacciones = pd.concat([transacciones, df_nuevos], ignore_index=True)

# Recalcular transaction_id para que sea único
transacciones['transaction_id'] = range(1, len(transacciones) + 1)

print(f'Se añadieron {len(df_nuevos)} nuevas filas.')
display(transacciones.tail(3))

Se añadieron 6000 nuevas filas.


,transaction_id,user_id,fecha_transaccion,descripcion_transaccion,monto_transaccion,medio_pago,edad,sexo,ingreso_mensual,linea_credito,credito_utilizado,frecuencia_ahorro,categoria_gasto
125462,125463,2999,2025-07-20,Proveedora Escolar,1194.34,Credito,54,M,41035,30000,23407,Alta,Educacion
125463,125464,3000,2025-06-15,Interceramic,3103.85,Debito,46,F,58563,30000,13664,Media,Vivienda
125464,125465,3000,2025-07-20,Librerías Gandhi,1320.05,Debito,46,F,58563,30000,13664,Media,Educacion


In [ ]:
# ya aparecen las categorías que faltaban en el set original

print (transacciones['categoria_gasto'].unique())
print (f'Ya se encuentran las 8 categorías')

['Alimentacion' 'Salud' 'Transporte' 'Compras' 'Servicios'
 'Entretenimiento' 'Vivienda' 'Educacion']
Ya se encuentran las 8 categorías


A partir de este punto ya existen todas las categorías de gasto requeridas por la empresa. A continuación se guardará el nuevo dataset con las categorías añadidas como "transacciones_ok.csv"

In [ ]:
transacciones.to_csv('transacciones_ok.csv', index=False)

Se revisan las columnas del set de transacciones para determinar cuáles son de utilidad para el modelo de clasificación de lenguaje natural.

In [ ]:
transacciones.columns

Index(['transaction_id', 'user_id', 'fecha_transaccion',
       'descripcion_transaccion', 'monto_transaccion', 'medio_pago', 'edad',
       'sexo', 'ingreso_mensual', 'linea_credito', 'credito_utilizado',
       'frecuencia_ahorro', 'categoria_gasto'],
      dtype='object')

Se determina eliminar las columnas correspondientes a edad y sexo del cliente, las de ingreso mensual, línea de crédito, crédito utilizado y frecuencia de ahorro, toda vez que son redundantes con la tabla de clientes o carecen de relevancia para el clasificador de procesamiento de lenguaje natural, asignándole la variables de "transacciones_ok".

In [ ]:
# Eliminando las variables que no tienen utilidad para este dataset

transacciones_ok = transacciones.drop(['edad', 'sexo', 'ingreso_mensual', 'linea_credito', 'credito_utilizado', 'frecuencia_ahorro'], axis = 1)

In [ ]:
# Revisando las columnas que se encuentran en transacciones_ok

transacciones_ok.columns

Index(['transaction_id', 'user_id', 'fecha_transaccion',
       'descripcion_transaccion', 'monto_transaccion', 'medio_pago',
       'categoria_gasto'],
      dtype='object')

In [ ]:
# Formateando todo el texto del set a minúsculas

text_columns = transacciones_ok.select_dtypes(include=['object']).columns

for col in text_columns:
    transacciones_ok[col] = transacciones_ok[col].str.lower()

display(transacciones_ok.head())

,transaction_id,user_id,fecha_transaccion,descripcion_transaccion,monto_transaccion,medio_pago,categoria_gasto
0,1,1,2025-10-30,soriana,1072.15,debito,alimentacion
1,2,1,2025-04-22,soriana,604.67,debito,alimentacion
2,3,1,2025-04-12,farmacias del ahorro,1798.57,transferencia,salud
3,4,1,2025-08-18,uber,1485.49,debito,transporte
4,5,1,2025-12-24,oxxo,1073.86,efectivo,alimentacion


In [ ]:
# Revisando el tipo de dato de cada columna

transacciones_ok.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 125465 entries, 0 to 125464
Data columns (total 7 columns):
 #   Column                   Non-Null Count   Dtype  
---  ------                   --------------   -----  
 0   transaction_id           125465 non-null  int64  
 1   user_id                  125465 non-null  int64  
 2   fecha_transaccion        125465 non-null  object 
 3   descripcion_transaccion  125465 non-null  object 
 4   monto_transaccion        125465 non-null  float64
 5   medio_pago               125465 non-null  object 
 6   categoria_gasto          125465 non-null  object 
dtypes: float64(1), int64(2), object(4)
memory usage: 6.7+ MB


In [ ]:
# Buscando valores nulos

transacciones_ok.isnull().sum()

,0
transaction_id,0
user_id,0
fecha_transaccion,0
descripcion_transaccion,0
monto_transaccion,0
medio_pago,0
categoria_gasto,0


In [ ]:
# Buscando valores duplicados

transacciones_ok.duplicated().sum()

np.int64(0)

NO EXISTEN DATOS NULOS, DUPLICADOS, NI EN MAYÚSCULAS

El set de transacciones se encuentra listo para análisis exploratorios y entrenamiento de modelos de aprendizaje automático.

# Revisión y limpieza de datos de la tabla Clientes

In [ ]:
# Se asigna una variable a la tabla y se revisan brevemente las columnas
clientes = pd.read_csv('/content/users.csv')
clientes.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3000 entries, 0 to 2999
Data columns (total 12 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   user_id                3000 non-null   int64  
 1   edad                   3000 non-null   int64  
 2   sexo                   3000 non-null   object 
 3   estado_civil           3000 non-null   object 
 4   numero_hijos           3000 non-null   int64  
 5   empleo_formal          3000 non-null   int64  
 6   ingreso_mensual        3000 non-null   int64  
 7   linea_credito          3000 non-null   int64  
 8   credito_utilizado      3000 non-null   int64  
 9   frecuencia_ahorro      3000 non-null   object 
 10  monto_promedio_ahorro  3000 non-null   float64
 11  perfil_financiero      3000 non-null   object 
dtypes: float64(1), int64(7), object(4)
memory usage: 281.4+ KB


In [ ]:
# Se revisa brevemente el contenido de la tabla

clientes.sample(5)

,user_id,edad,sexo,estado_civil,numero_hijos,empleo_formal,ingreso_mensual,linea_credito,credito_utilizado,frecuencia_ahorro,monto_promedio_ahorro,perfil_financiero
158,159,43,M,Soltero,1,0,15108,30000,16456,Media,1510.80,En observacion
2305,2306,66,M,Casado,2,0,12516,15000,4695,Alta,2753.52,En observacion
119,120,69,F,Divorciado,4,0,62018,50000,8618,Media,6201.80,En observacion
118,119,22,F,Divorciado,0,0,24581,15000,1425,Baja,737.43,En observacion
1700,1701,25,F,Unión libre,2,0,79024,120000,15418,Media,7902.40,En observacion


Se trata de una tabla de 12 columnas y 3,000 filas. Las columnas se dividen entre el identificador de cliente, edad del cliente, sexo del cliente, estado civil del cliente, número de hijos, si cuenta o no cuenta con un empleo formal, el ingreso mensual del cliente expresado en pesos mexicanos (MXN), línea de crédito del cliente en pesos mexicanos, el crédito ocupado por el cliente en pesos mexicanos, la frecuencia de ahorro del cliente presumiblemente determinada de acuerdo con el resultado entre sus ingresos, el crédito utilizado y otras erogaciones del cliente no rastreables de forma digital como retiros de efectivo, el monto promedio de ahorro y el perfil financiero del cliente, clasificado entre saludable, en observación y en riesgo.

In [ ]:
# Se revisan brevemente las columnas de la tabla de clientes

clientes.columns

Index(['user_id', 'edad', 'sexo', 'estado_civil', 'numero_hijos',
       'empleo_formal', 'ingreso_mensual', 'linea_credito',
       'credito_utilizado', 'frecuencia_ahorro', 'monto_promedio_ahorro',
       'perfil_financiero'],
      dtype='object')

In [ ]:
# Formateando todo el texto del set a minúsculas

text_columns_clientes = clientes.select_dtypes(include=['object']).columns

for col in text_columns_clientes:
    clientes[col] = clientes[col].str.lower()

display(clientes.head())

,user_id,edad,sexo,estado_civil,numero_hijos,empleo_formal,ingreso_mensual,linea_credito,credito_utilizado,frecuencia_ahorro,monto_promedio_ahorro,perfil_financiero
0,1,58,m,soltero,2,0,37256,30000,24132,baja,1117.68,en riesgo
1,2,31,m,divorciado,1,1,61354,30000,3090,baja,1840.62,en observacion
2,3,29,f,divorciado,1,1,28866,15000,6267,alta,6350.52,en observacion
3,4,19,f,unión libre,3,0,69261,30000,20390,media,6926.10,en riesgo
4,5,62,m,divorciado,0,0,45268,80000,62746,media,4526.80,en riesgo


In [ ]:
# Buscando valores nulos en la tabla de clientes

clientes.isnull().sum()

,0
user_id,0
edad,0
sexo,0
estado_civil,0
numero_hijos,0
empleo_formal,0
ingreso_mensual,0
linea_credito,0
credito_utilizado,0
frecuencia_ahorro,0


In [ ]:
# Buscando valores duplicados en la tabla de clientes

clientes.duplicated().sum()

np.int64(0)

NO EXISTEN VALORES NULOS NI DUPLICADOS EN EL DATASET DE CLIENTES.

El set de clientes se encuentra listo para entrenamiento de modelos de aprendizaje automático de clasificación.

# Uniendo ambos datasets

In [ ]:
# Realizando el merge de los datasets por user_id
df_final = pd.merge(transacciones_ok, clientes, on='user_id', how='inner')

# Verificando el resultado
print(f'Dimensiones del nuevo dataset: {df_final.shape}')
display(df_final.sample(5))

Dimensiones del nuevo dataset: (125465, 18)


,transaction_id,user_id,fecha_transaccion,descripcion_transaccion,monto_transaccion,medio_pago,categoria_gasto,edad,sexo,estado_civil,numero_hijos,empleo_formal,ingreso_mensual,linea_credito,credito_utilizado,frecuencia_ahorro,monto_promedio_ahorro,perfil_financiero
48621,48622,1218,2025-11-04,spotify,1635.91,credito,entretenimiento,65,m,unión libre,4,0,58269,15000,10045,baja,1748.07,en riesgo
44172,44173,1107,2025-05-20,walmart,877.07,debito,alimentacion,41,m,divorciado,0,1,13226,50000,30847,media,1322.60,en riesgo
101266,101267,2541,2025-08-13,telcel,1430.20,credito,servicios,46,m,soltero,4,1,49106,30000,27944,baja,1473.18,en riesgo
83886,83887,2107,2025-08-29,walmart,492.22,transferencia,alimentacion,53,f,unión libre,0,0,19687,30000,3934,baja,590.61,en observacion
48504,48505,1215,2025-03-12,pemex,206.81,credito,transporte,31,m,casado,2,1,73836,30000,1784,media,7383.60,en observacion


Se trata de la unión en identificador de cliente de las tablas de transacciones y clientes, la tabla nueva consta de 18 columnas y 125,465 filas.